In [0]:
-- Test 1: Da li postoje sve dimension tabele?
SHOW TABLES IN acme_catalog.gold LIKE 'dim_%';

show tables in acme_catalog.gold;

drop table acme_catalog.gold.fact_sales;

-- Test 2: Da li postoje sve fact tabele?
SHOW TABLES IN acme_catalog.gold LIKE 'fact_%';

-- Test 3: Da li dim_date ima YoY polja?
DESCRIBE TABLE acme_catalog.gold.dim_date;

-- Test 4: Da li dim_product ima product_line?
select * from acme_catalog.gold.dim_products;

SELECT DISTINCT category_id, category_name 
FROM acme_catalog.gold.dim_products 
WHERE is_current = TRUE;

-- Test 5: Da li freight postoji samo u fact_orders?
SELECT COUNT(*) as freight_in_orders 
FROM acme_catalog.gold.fact_orders 
WHERE freight IS NOT NULL;

SELECT COUNT(*) as freight_in_lines 
FROM acme_catalog.gold.fact_order_lines 
WHERE freight IS NOT NULL;  -- Treba biti 0!

-- Test 6: Da li su measures kalkulisane?
SELECT order_id, line_no, quantity, unit_price, discount,
       gross_sales, net_sales, gross_margin, margin_percent
FROM acme_catalog.gold.fact_order_lines
LIMIT 5;

describe table acme_catalog.gold.fact_order_lines;


-- Analyze sales figures and compare them year-over-year.

select * from acme_catalog.gold.fact_order_lines;

select * from acme_catalog.gold.fact_orders;

select * from acme_catalog.gold.fact_shipments;

WITH current_year_sales AS (
  SELECT 
    d.year, d.month, d.month_name,
    SUM(fol.revenue) as total_revenue,
    SUM(fol.margin) as total_margin,
    COUNT(DISTINCT fo.order_id) as order_count
  FROM acme_catalog.gold.fact_order_lines fol
  JOIN acme_catalog.gold.fact_orders fo ON fol.order_id = fo.order_id
  JOIN acme_catalog.gold.dim_date d ON fo.date_id = d.date_id
  GROUP BY d.year, d.month, d.month_name
),
last_year_sales AS (
  SELECT 
    d.year + 1 as year, d.month,
    SUM(fol.revenue) as ly_revenue,
    SUM(fol.margin) as ly_margin
  FROM acme_catalog.gold.fact_order_lines fol
  JOIN acme_catalog.gold.fact_orders fo ON fol.order_id = fo.order_id
  JOIN acme_catalog.gold.dim_date d ON fo.date_id = d.date_id
  GROUP BY d.year, d.month
)
SELECT 
  cy.year, cy.month, cy.month_name,
  cy.total_revenue as current_revenue,
  ly.ly_revenue as last_year_revenue,
  cy.total_revenue - COALESCE(ly.ly_revenue, 0) as yoy_revenue_change,
  ROUND(((cy.total_revenue - COALESCE(ly.ly_revenue, 0)) / NULLIF(ly.ly_revenue, 0)) * 100, 2) as yoy_revenue_pct
FROM current_year_sales cy
LEFT JOIN last_year_sales ly ON cy.year = ly.year AND cy.month = ly.month
ORDER BY cy.year DESC, cy.month DESC;

-- Measure and track sales margins.
-- Analyze sales data by various dimensions (e.g., Region, Country, Division, Product Line, Product Group, and Customer).
-- Compute average sales per transaction and per customer.
-- Follow up on customer growth over time (e.g., Year-To-Date vs. Last Year-To-Date).

-- agg_sales_by_month
select
        dd.year, dd.quarter, dd.month, dd.month_name, dc.division_id, dp.category_id,
        cast(sum(fol.revenue) as decimal(14,2)) as total_revenue,
        cast(sum(fol.cost) as decimal(14,2)) as total_cost,
        cast(sum(fol.margin) as decimal(14,2)) as total_margin,
        count(distinct fo.order_id) as order_count,
        count(distinct fo.customer_sk) as distinct_customers
    from acme_catalog.gold.fact_order_lines fol
    join acme_catalog.gold.fact_orders fo on fol.order_id = fo.order_id
    join acme_catalog.gold.dim_date dd on fo.date_id = dd.date_id
    join acme_catalog.gold.dim_customers dc on fo.customer_sk = dc.customer_sk and dc.is_current = true
    join acme_catalog.gold.dim_products dp on fol.product_sk = dp.product_sk and dp.is_current = true
    group by dd.year, dd.quarter, dd.month, dd.month_name, dc.division_id, dp.category_id

-- agg_customer_growth_by_month
with monthly_customers as (
        select
            dd.year, dd.quarter, dd.month, dd.month_name, fo.customer_sk,
            min(dd.full_date) over (
                partition by fo.customer_sk order by dd.full_date
                rows between unbounded preceding and unbounded following
            ) as first_order_date
        from {GOLD}.fact_orders fo
        join {GOLD}.dim_date dd on fo.date_id = dd.date_id
    ),
    monthly_agg as (
        select
            year, quarter, month, month_name,
            count(distinct customer_sk) as active_customers,
            count(distinct case
                when date_format(first_order_date, 'yyyyMM') = concat(year, lpad(month, 2, '0'))
                then customer_sk
            end) as new_customers
        from monthly_customers
        group by year, quarter, month, month_name
    )
    select
        year, quarter, month, month_name, new_customers, active_customers,
        sum(new_customers) over (
            partition by year order by month
            rows between unbounded preceding and current row
        ) as ytd_customers
    from monthly_agg

select * from acme_catalog.gold.agg_sales_by_month;

select * from acme_catalog.gold.agg_customer_growth_by_month;